# Word Embeddings: Meaning in Numbers
**COMP 395 — Deep Learning | In-Class Activity**

In this notebook you will:
1. Load pre-trained GloVe word embeddings
2. Inspect what a single embedding vector looks like
3. Measure word similarity with cosine similarity
4. Test the famous king − man + woman ≈ queen analogy
5. Visualize embeddings in 2D with PCA
6. Explore gender bias in word embeddings

---

## Setup

We use `gensim` to load pre-trained GloVe vectors. Run this cell first — the download is ~66 MB and may take a minute.

In [ ]:
# Install gensim if needed (uncomment the line below)
# !pip install gensim

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import gensim.downloader as api

# Load GloVe vectors (50 dimensions, trained on Wikipedia + Gigaword)
print("Loading GloVe embeddings (this may take a minute)...")
glove = api.load("glove-wiki-gigaword-50")
print(f"Loaded {len(glove)} word vectors, each with {glove.vector_size} dimensions.")

---
## Part 1: What Does a Single Embedding Look Like?

Remember: `nn.Embedding` is a lookup table. Given an integer index, it returns a dense vector.  
GloVe works the same way — given a *word*, it returns a vector of 50 floats.

In [ ]:
# Look up the embedding for a single word
word = "queen"
vec = glove[word]

print(f"Word: '{word}'")
print(f"Shape: {vec.shape}")
print(f"Type: {type(vec)}")
print(f"\nFirst 10 values: {vec[:10]}")
print(f"Min: {vec.min():.3f}, Max: {vec.max():.3f}, Mean: {vec.mean():.3f}")

In [ ]:
# Visualize the embedding as a heatmap
fig, ax = plt.subplots(figsize=(12, 1.2))
im = ax.imshow(vec.reshape(1, -1), cmap="RdBu_r", aspect="auto", vmin=-3, vmax=3)
ax.set_xlabel("Dimension")
ax.set_yticks([])
ax.set_title(f'Embedding for "{word}" — 50 learned floats', fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

**Quick check:** No single dimension means "royalty" or "gender." The meaning is *distributed* across all 50 numbers. This is why we call them *distributed representations*.

---
## Part 2: Cosine Similarity

How do we measure whether two words are similar? We use **cosine similarity** — the cosine of the angle between two vectors:

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \; \|\mathbf{b}\|}$$

- Close to **1** → very similar
- Close to **0** → unrelated
- Close to **-1** → opposite

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Try some word pairs
pairs = [
    ("cat", "dog"),
    ("cat", "car"),
    ("king", "queen"),
    ("king", "pizza"),
    ("happy", "sad"),
    ("happy", "joyful"),
]

print(f"{'Word Pair':<25} {'Cosine Similarity':>18}")
print("-" * 45)
for w1, w2 in pairs:
    sim = cosine_similarity(glove[w1], glove[w2])
    print(f"{w1 + ' ↔ ' + w2:<25} {sim:>18.4f}")

In [ ]:
# gensim has a built-in method — find the most similar words
print("Words most similar to 'python':")
for word, score in glove.most_similar("python", topn=8):
    print(f"  {word:<15} {score:.4f}")

**Try it yourself:** Pick 2–3 word pairs and check their similarity. Any surprises?

In [ ]:
# YOUR EXPLORATION HERE
# Example: cosine_similarity(glove["YOUR_WORD"], glove["OTHER_WORD"])


### Aside: Calibrating Your Intuition

When you see a cosine similarity of 0.5, is that high or low? To answer that, we need a **baseline**.

In high-dimensional spaces, something surprising happens: random vectors are almost always **nearly orthogonal** (cosine similarity ≈ 0). This is called **concentration of measure** — as dimensionality $d$ grows, the expected cosine similarity between two random unit vectors is 0, with variance ≈ $1/d$.

At $d = 50$, the standard deviation is only ~0.14. So a cosine similarity of 0.5 between "cat" and "dog" is **3+ standard deviations** above what random chance would give you. That's not mediocre — that's a strong signal.

Let's verify this empirically:

In [ ]:
# Generate random 50-d vectors and check their pairwise cosine similarities
np.random.seed(42)
n_vectors = 1000
d = 50

# Random unit vectors
random_vecs = np.random.randn(n_vectors, d)
random_vecs = random_vecs / np.linalg.norm(random_vecs, axis=1, keepdims=True)

# Compute pairwise cosine similarities (sample 10,000 random pairs)
n_pairs = 10_000
idx_a = np.random.randint(0, n_vectors, n_pairs)
idx_b = np.random.randint(0, n_vectors, n_pairs)
mask = idx_a != idx_b  # don't compare a vector to itself
idx_a, idx_b = idx_a[mask], idx_b[mask]

random_sims = np.array([
    np.dot(random_vecs[a], random_vecs[b]) for a, b in zip(idx_a, idx_b)
])

# Now compare: real GloVe similarities
real_pairs = [("cat", "dog"), ("king", "queen"), ("happy", "joyful")]
real_sims = [cosine_similarity(glove[a], glove[b]) for a, b in real_pairs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(random_sims, bins=60, density=True, alpha=0.7, color="#AAAAAA", label="Random 50-d vectors")
for (w1, w2), sim in zip(real_pairs, real_sims):
    ax.axvline(sim, color="#C0392B", linewidth=2, linestyle="--")
    ax.text(sim, ax.get_ylim()[1] * 0.85, f"{w1}-{w2}\n{sim:.2f}",
            ha="center", fontsize=9, fontweight="bold", color="#C0392B")

ax.set_xlabel("Cosine Similarity", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title(f"Random vectors in d={d}: mean={random_sims.mean():.3f}, std={random_sims.std():.3f}", fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(-0.6, 1.0)
plt.tight_layout()
plt.show()

print(f"Theoretical std: {1/np.sqrt(d):.3f}")
print(f"Empirical std:   {random_sims.std():.3f}")
print(f"\nReal word similarities are WAY out in the tail — these aren't accidents.")

> **Fine print for the math-curious:** Cosine similarity is not a proper distance metric — cosine *distance* ($1 - \cos$) doesn't satisfy the **triangle inequality** ($d(a,c) \leq d(a,b) + d(b,c)$). If you want a real metric, use angular distance ($\frac{1}{\pi}\arccos$). The triangle inequality is one of those things you prove in real analysis and then spend the rest of your career appreciating. Take real analysis.

---
## Part 3: Vector Arithmetic — The King–Queen Analogy

One of the most famous results from word embeddings:

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

The idea: subtracting "man" removes the male component, adding "woman" inserts the female component, and the royalty component stays.

In [ ]:
# Manual vector arithmetic
result_vec = glove["king"] - glove["man"] + glove["woman"]

# Find the closest word to our result vector
# (gensim's most_similar can take a positive/negative word list)
print("king - man + woman =")
for word, score in glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5):
    print(f"  {word:<15} {score:.4f}")

In [ ]:
# Try more analogies!
analogies = [
    (["paris", "germany"], ["france"], "paris - france + germany = ?"),
    (["bigger", "cold"], ["big"], "bigger - big + cold = ?"),
    (["king", "woman"], ["man"], "king - man + woman = ?"),
]

for pos, neg, description in analogies:
    result = glove.most_similar(positive=pos, negative=neg, topn=3)
    top_word, top_score = result[0]
    print(f"{description:<35} → {top_word} ({top_score:.3f})")

---
## Part 4: Visualizing Embeddings with PCA

50 dimensions is too many to visualize. We'll use **PCA** (which you already know!) to project down to 2D.

Remember: PCA finds the directions of maximum variance. We lose information, but the relative positions are approximately preserved.

In [ ]:
# Choose semantically grouped words
word_groups = {
    "Animals":  ["cat", "dog", "fish", "bird", "horse"],
    "Royalty":  ["king", "queen", "prince", "princess", "throne"],
    "Tech":     ["computer", "software", "algorithm", "network", "data"],
    "Food":     ["pizza", "pasta", "bread", "rice", "salad"],
}

colors = {"Animals": "#2A62B0", "Royalty": "#E07020", "Tech": "#2E8B57", "Food": "#C0392B"}

# Collect all words and their vectors
all_words = []
all_vecs = []
all_labels = []
for group, words in word_groups.items():
    for w in words:
        all_words.append(w)
        all_vecs.append(glove[w])
        all_labels.append(group)

all_vecs = np.array(all_vecs)
print(f"Matrix shape: {all_vecs.shape}  (words × dimensions)")

In [ ]:
# PCA: 50 dims → 2 dims
pca = PCA(n_components=2)
coords = pca.fit_transform(all_vecs)

print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.1%}")

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

for i, (word, group) in enumerate(zip(all_words, all_labels)):
    ax.scatter(coords[i, 0], coords[i, 1], color=colors[group], s=80, zorder=3)
    ax.annotate(word, (coords[i, 0], coords[i, 1]),
                fontsize=10, fontweight="bold",
                xytext=(5, 5), textcoords="offset points")

# Legend
for group, color in colors.items():
    ax.scatter([], [], color=color, s=80, label=group)
ax.legend(fontsize=11, loc="best")

ax.set_xlabel("PC1", fontsize=12)
ax.set_ylabel("PC2", fontsize=12)
ax.set_title("GloVe Embeddings Projected to 2D with PCA", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Discuss with your partner:** Do the clusters make sense? Are any words positioned in unexpected places?

---
## Part 5: Bias in Word Embeddings

Bolukbasi et al. (2016) showed that word embeddings trained on Google News text encode gender stereotypes. The key insight: there is a **gender direction** in embedding space, and occupation words are not neutral along this direction.

Let's replicate a simplified version of their analysis.

In [ ]:
# Step 1: Define a "gender direction" as she - he
gender_direction = glove["she"] - glove["he"]
gender_direction = gender_direction / np.linalg.norm(gender_direction)  # unit vector

print(f"Gender direction shape: {gender_direction.shape}")
print(f"This is a unit vector pointing from 'he' toward 'she' in embedding space.")

In [ ]:
# Step 2: Project occupation words onto the gender direction
# A positive projection → closer to "she", negative → closer to "he"
occupations = [
    "programmer", "engineer", "scientist", "doctor", "lawyer",
    "nurse", "teacher", "homemaker", "receptionist", "librarian",
    "architect", "surgeon", "pilot", "dancer", "secretary",
    "professor", "athlete", "journalist", "chef", "therapist",
]

# Compute projection (dot product with gender direction)
projections = {}
for occ in occupations:
    proj = np.dot(glove[occ], gender_direction)
    projections[occ] = proj

# Sort by projection value
sorted_occs = sorted(projections.items(), key=lambda x: x[1])

print(f"{'Occupation':<18} {'Projection':>10}   Direction")
print("-" * 45)
for occ, proj in sorted_occs:
    arrow = "← he" if proj < 0 else "she →"
    bar = "█" * int(abs(proj) * 15)
    print(f"{occ:<18} {proj:>10.4f}   {arrow} {bar}")

In [ ]:
# Step 3: Visualize the bias
fig, ax = plt.subplots(figsize=(10, 6))

occs = [item[0] for item in sorted_occs]
projs = [item[1] for item in sorted_occs]
bar_colors = ["#2A62B0" if p < 0 else "#D14D72" for p in projs]

ax.barh(range(len(occs)), projs, color=bar_colors, edgecolor="white", height=0.7)
ax.set_yticks(range(len(occs)))
ax.set_yticklabels(occs, fontsize=10)
ax.set_xlabel("Projection onto gender direction (she − he)", fontsize=11)
ax.set_title("Gender Bias in GloVe Occupation Embeddings", fontsize=13)
ax.axvline(x=0, color="black", linewidth=0.8)

# Labels
ax.text(min(projs) * 0.5, len(occs) + 0.5, "← he direction",
        fontsize=10, color="#2A62B0", fontweight="bold", ha="center")
ax.text(max(projs) * 0.5, len(occs) + 0.5, "she direction →",
        fontsize=10, color="#D14D72", fontweight="bold", ha="center")

plt.tight_layout()
plt.show()

### Reflection

These biases weren't programmed in — they were **learned from text**. The training corpus (Wikipedia, news articles) reflects societal patterns, and the embeddings absorb them.

**Discuss:** If a hiring tool uses these embeddings to match resumes to job descriptions, what could go wrong?

---
## Connection to `nn.Embedding`

In PyTorch, `nn.Embedding` is the same lookup table structure:

```
Text → tokens → integer indices → nn.Embedding → dense vectors → model
```

- **Pre-trained:** Load GloVe/Word2Vec weights into `nn.Embedding` — start with general language knowledge
- **Learned from scratch:** Initialize randomly, train with your task — embeddings adapt to your data

In your next lab, you will **learn embeddings from scratch** as part of training an RNN. The vectors will start random and gradually organize themselves — just like GloVe, but specialized to your task.

---
## Exit Ticket

**Submit your answers before leaving class.**

1. A GloVe embedding for the word "pizza" is a vector of 50 floats. In one sentence, what does this vector *represent*?

2. Why is cosine similarity preferred over Euclidean distance for comparing word embeddings? (1–2 sentences)

3. If `nn.Embedding(10000, 64)` creates an embedding layer, what is the shape of its internal weight matrix? What does each *row* represent?

4. Give one concrete example of how bias in word embeddings could cause harm in a deployed system.